# E1 Colab A100 lock → Braket

Compress Experiment Set 1 (E1–E4) into one GPU lock, then stop Colab and move to Braket.
Do **not** re-run the full 324-point photonic campaign here.

In [ ]:
import jax, pathlib, subprocess, sys
print(jax.__version__, jax.default_backend(), jax.devices())
REPO = pathlib.Path('.').resolve()
print('REPO', REPO)

## Phase 1 — E1–E4 lock (require GPU on A100)

In [ ]:
subprocess.run([
    sys.executable, 'experiments/e1_colab_lock.py',
    '--output-dir', 'docs/benchmark-results',
    '--require-gpu',
], cwd=REPO, check=True)

## Phase 2a — Braket SV1 smoke
Use `--emulate` until AWS budgets/credentials are ready; then `--live` with `BRAKET_SUBMIT=1`.

In [ ]:
import os
cmd = [sys.executable, 'experiments/braket_live_smoke.py',
       '--output-dir', 'docs/benchmark-results/braket-live']
if os.environ.get('BRAKET_SUBMIT') == '1' and os.environ.get('BRAKET_S3_BUCKET'):
    cmd.append('--live')
else:
    cmd.append('--emulate')
subprocess.run(cmd, cwd=REPO, check=True)

## Phase 2b — Hybrid E1–E4 (local equivalent, then optional submit packaging)

In [ ]:
subprocess.run([
    sys.executable, 'experiments/braket_hybrid_e1_e4.py',
    '--output-dir', 'docs/benchmark-results/braket-live',
    '--reference-lock', 'docs/benchmark-results/e1_colab_lock.json',
    '--local-hybrid',
], cwd=REPO, check=True)